# Notebook 05 — Adversarial Rewriting on Google Colab

**UPLOAD AND RUN THIS ON GOOGLE COLAB — NOT LOCALLY**

**Project:** MSc AI Dissertation — AI-Generated Text Detection  
**Student:** Abdul Hannaan Mohammed | B00409227 | UWS  

**Model:** `Vamsi/T5_Paraphrase_Paws`  
T5 fine-tuned on the PAWS paraphrase dataset. Size ~900MB. Works on Colab T4.

**Before running:** Runtime > Change runtime type > T4 GPU

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/MSc-AI-Detection/adversarial'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted.')
print(f'Output folder: {DRIVE_OUTPUT}')

## Cell 2 — Upload CSV

A file picker will appear. Select `ai_samples_500_for_colab.csv` from your local machine.

In [ ]:
from google.colab import files
import pandas as pd
import io

print('Select ai_samples_500_for_colab.csv from your local machine...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'Loaded : {filename}')
print(f'Rows   : {len(df)}')
print(f'Preview: {df["text"].iloc[0][:150]}')

## Cell 3 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    DEVICE = 'cpu'
    print('No GPU — running on CPU')

print(f'Device: {DEVICE}')

## Cell 4 — Install Libraries

In [ ]:
!pip install -q transformers sentencepiece
print('Done.')

## Cell 5 — Load Model

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME = 'Vamsi/T5_Paraphrase_Paws'

print(f'Loading {MODEL_NAME}...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print(f'Model loaded on {DEVICE}')

## Cell 6 — Define Paraphrase Function and Test

**Must pass before running Cell 7.**

In [ ]:
def paraphrase(text):
    # Use first 80 words — model works best on shorter input
    words = str(text).strip().split()
    short = ' '.join(words[:80])

    # Required input format for this model
    input_text = f'paraphrase: {short} </s>'

    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=256,
        truncation=True
    )

    input_ids      = inputs['input_ids'].to(DEVICE)
    attention_mask = inputs['attention_mask'].to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256,
            do_sample=True,
            top_k=200,
            top_p=0.95,
            num_return_sequences=1
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Test on 3 samples before the full run
print('=== TEST RESULTS ===')
for i in range(3):
    orig = df['text'].iloc[i]
    rewr = paraphrase(orig)
    print(f'\n--- Sample {i+1} ---')
    print(f'ORIGINAL : {orig[:200]}')
    print(f'REWRITTEN: {rewr}')

print('\n=== TEST PASSED — run Cell 7 ===')

## Cell 7 — Rewrite All 500 Samples

Expected time: 20–40 minutes on T4 GPU.  
Saves a checkpoint to Google Drive every 50 samples.

> SCREENSHOT: Take a screenshot while this runs showing the progress lines.  
> Save as: screenshots/12_paraphraser_colab_running.png

In [ ]:
import time

OUTPUT_FILE      = os.path.join(DRIVE_OUTPUT, 'pegasus_rewritten_500.csv')
CHECKPOINT_EVERY = 50

results     = []
error_count = 0
start_time  = time.time()

print(f'Rewriting {len(df)} samples...')
print(f'Model : Vamsi/T5_Paraphrase_Paws')
print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_FILE}')
print('-' * 50)

for i, row in df.iterrows():
    original  = str(row['text'])
    rewritten = original
    success   = True

    try:
        rewritten = paraphrase(original)
    except Exception as e:
        success = False
        error_count += 1
        if error_count <= 5:
            print(f'  ERROR sample {i}: {type(e).__name__}: {str(e)[:100]}')

    results.append({
        'original_text':  original,
        'rewritten_text': rewritten,
        'label':          1,
        'label_name':     'ai',
        'success':        success
    })

    done = i + 1
    if done % 10 == 0:
        elapsed   = time.time() - start_time
        per_item  = elapsed / done
        remaining = per_item * (len(df) - done)
        print(f'  [{done:3d}/{len(df)}]  '
              f'Elapsed: {elapsed/60:.1f}m  '
              f'ETA: {remaining/60:.1f}m  '
              f'Errors: {error_count}')

    if done % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        print(f'  >>> Checkpoint saved: {done} samples')

# Final save
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_FILE, index=False)

elapsed    = time.time() - start_time
successful = int(results_df['success'].sum())

print('\n' + '=' * 50)
print('REWRITING COMPLETE')
print(f'  Total      : {len(results_df)}')
print(f'  Successful : {successful}')
print(f'  Errors     : {error_count}')
print(f'  Time       : {elapsed/60:.1f} minutes')
print(f'  Saved to   : {OUTPUT_FILE}')
print('=' * 50)

## Cell 8 — Preview Results

> SCREENSHOT: Screenshot this output.  
> Save as: screenshots/13_paraphrasing_complete.png

In [ ]:
ok = results_df[results_df['success'] == True]
print(f'Successful rewrites: {len(ok)} / {len(results_df)}')
print()

for i in range(min(3, len(ok))):
    row = ok.iloc[i]
    print(f'--- Example {i+1} ---')
    print(f'ORIGINAL : {row["original_text"][:250]}')
    print(f'REWRITTEN: {row["rewritten_text"][:250]}')
    print()

## Cell 9 — Download Output File

After downloading, move the file to: `data/adversarial/pegasus_rewritten_500.csv`

In [ ]:
files.download(OUTPUT_FILE)
print('Download started.')
print('Move file to: data/adversarial/pegasus_rewritten_500.csv')